# 05 · FT-Transformer (정형 데이터용 Transformer)

> 역할: MLP보다 강한 딥러닝으로, **각 피처를 토큰으로 보는 Transformer**를 쓴다.
> 핵심 질문: MLP(PR_AUC 0.193)와 트리(XGBoost 0.244) 사이 어디에 위치하나?
> 올바른 DL 구조가 트리와의 격차를 줄일 수 있는지 본다.

## 0. 환경 설정
- **torch**: 신경망 프레임워크. 여기선 내장 `nn.TransformerEncoder`를 그대로 활용해 직접 attention을 짜지 않고도 Transformer를 구성한다.
- 04와 같은 데이터(load_processed)·같은 평가(compute_metrics)를 써서 공정 비교.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from utils import set_seed, compute_metrics, log_result, load_processed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
OUT_DIR     = PROJECT_ROOT / "processed"
RESULTS_CSV = PROJECT_ROOT / "notebooks_v1" / "results.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. 데이터 불러오기
02에서 만든 데이터를 float32 배열로.

In [ ]:
train_df, val_df, test_df = load_processed(OUT_DIR)
TARGET = "went_on_backorder"
feature_cols = [c for c in train_df.columns if c != TARGET]

X_train = train_df[feature_cols].values.astype("float32")
y_train = train_df[TARGET].values.astype("float32")
X_val   = val_df[feature_cols].values.astype("float32")
y_val   = val_df[TARGET].values.astype("float32")

print("피처 수:", len(feature_cols))

## 2. 배치로 묶기
Transformer는 MLP보다 무거워서 val도 한 번에 안 올리고 배치로 나눠 예측한다(메모리 안전).

In [ ]:
def make_loader(X, y, batch_size=4096, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val, y_val, shuffle=False)

## 3. FT-Transformer 모델 - 왜 정형에 Transformer를 쓰나
LSTM은 '시간 순서'가 필요했지만(우리 데이터엔 없음), Transformer의 self-attention은
**순서 없는 토큰 집합**에서 '어느 변수와 어느 변수가 함께 중요한지'(상호작용)를 학습한다.
그래서 각 피처(컬럼)를 토큰으로 만들면 정형 데이터에 딱 맞는다.

구조:
1. **피처 토크나이저**: 각 피처값(스칼라)을 d_token 차원 벡터로 바꾼다 (값 x 학습벡터 + 편향).
2. **[CLS] 토큰**: 전체를 요약할 학습용 토큰을 맨 앞에 붙인다 (BERT와 같은 아이디어).
3. **Transformer 인코더**: 토큰들끼리 attention으로 상호작용을 학습.
4. **[CLS] 출력 -> 선형층**: 백오더 점수 1개.
위치 인코딩은 안 쓴다 (피처는 순서 없는 집합이라).

In [ ]:
class FTTransformer(nn.Module):
    def __init__(self, num_features, d_token=32, n_layers=3, n_heads=4, dropout=0.1):
        super().__init__()
        # 1) 피처 토크나이저: 각 피처(스칼라) -> d_token 차원 벡터
        self.weight = nn.Parameter(torch.randn(num_features, d_token) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(num_features, d_token))
        # 2) [CLS] 토큰: 전체 요약용 학습 토큰
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        # 3) Transformer 인코더 (파이토치 내장)
        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 2,
            dropout=dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        # 4) [CLS] -> 점수 1개
        self.head = nn.Linear(d_token, 1)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.weight + self.bias   # (batch, F, d_token)
        cls = self.cls.expand(x.size(0), -1, -1)             # (batch, 1, d_token)
        seq = torch.cat([cls, tokens], dim=1)                # (batch, F+1, d_token)
        out = self.encoder(seq)                              # (batch, F+1, d_token)
        return self.head(out[:, 0, :]).squeeze(1)            # [CLS] -> (batch,)

## 4. Focal Loss
04 실험에서 Focal이 제일 좋았으므로 동일하게 사용 (공정 비교를 위해 불균형 전략 통일).

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

## 5. 학습 / 예측 함수
Transformer는 무거우니 에폭마다 진행상황을 찍는다. 예측은 배치로 모아서 한다.

In [ ]:
def predict(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for xb, yb in loader:
            out.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.concatenate(out)

def train_transformer(model, loader, epochs=6, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = FocalLoss()
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        print("  epoch", epoch + 1, "/", epochs, "완료")
    return model

## 6. 학습 + 평가 + 기록
CPU에서 Transformer는 매우 무겁다. 전체 135만 행은 에폭당 약 20분이라 비현실적이어서,
**학습은 무작위 일부(클래스 비율 유지)만 쓰고, 평가는 val 전체**로 한다.
이 '학습 비용이 트리/MLP보다 훨씬 크다'는 점 자체가, 정형 데이터에서 트리가 실무적으로
선호되는 이유 중 하나다 (보고서에 쓸 발견점).
GPU나 시간 여유가 있으면 SUBSAMPLE = None 으로 두면 전체를 쓴다.

In [ ]:
SUBSAMPLE = 400000   # 학습에 쓸 행 수 (None이면 전체)

if SUBSAMPLE is not None and SUBSAMPLE < len(X_train):
    rng = np.random.default_rng(42)
    sel = rng.choice(len(X_train), size=SUBSAMPLE, replace=False)  # 비율 그대로 유지
    fit_loader = make_loader(X_train[sel], y_train[sel], shuffle=True)
    print("학습 샘플:", len(sel), " 양성:", int(y_train[sel].sum()))
else:
    fit_loader = train_loader

set_seed(42)
model = FTTransformer(num_features=len(feature_cols), n_layers=2)
model = train_transformer(model, fit_loader, epochs=2)

val_prob = predict(model, val_loader)
metrics = compute_metrics(y_val, val_prob)
log_result("FT_Transformer", metrics, path=str(RESULTS_CSV))
print("FT-Transformer", metrics)

## 7. 저장 (분석용 결과 파일)
아래 두 파일은 '실행 파일'이 아니라 08_Decision_Analysis가 쓸 분석용 저장 결과다.

In [ ]:
torch.save(model.state_dict(), PROJECT_ROOT / "notebooks_v1" / "ft_transformer.pt")
np.save(PROJECT_ROOT / "notebooks_v1" / "ft_transformer_val_prob.npy", val_prob)
print("저장 완료: ft_transformer.pt, ft_transformer_val_prob.npy")

## 8. 결과 비교표
지금까지 모든 모델을 PR_AUC 내림차순으로 비교.

In [ ]:
res = pd.read_csv(RESULTS_CSV)
res = res.drop_duplicates(subset="model", keep="last")
res = res.sort_values("PR_AUC", ascending=False)
res[["model", "PR_AUC", "ROC_AUC", "Recall", "Precision", "F1"]]

---
### 정리
- FT-Transformer: 각 피처를 토큰으로 만들고 [CLS]로 요약하는, 정형 데이터에 맞는 Transformer.
- 핵심 비교: MLP(0.193)와 트리(XGBoost 0.244) 사이 어디인가?
  - 트리에 근접하면: 올바른 DL 구조가 격차를 줄인다는 증거.
  - 여전히 트리 아래면: 정형 스냅샷에선 트리가 강하다는 논지를 한 번 더 뒷받침.
- 다음: 06_TabNet(해석 가능한 정형 DL), 07_AutoEncoder(이상탐지 관점).

주의: Recall/F1은 threshold 0.5 기준이라 낮게 보임 -> 운영점·비용 기준은 08번에서.